# CellRank on the joint devbrain + Brooks object: moscot only

Velocity cannot participate here - Brooks ships no spliced/unspliced counts - so this
notebook runs the kernels that only need an embedding and a time label:
`RealTimeKernel` (moscot OT) and `ConnectivityKernel`.

What it buys over `cellrank_trajectory.ipynb`: the full **nine-stage** timeline
(E7.0, E7.5, E7.75, E8.0, E8.25, E8.5, E8.75, E9.0, E10.0) on a graph that is a single
connected component, which is the setting where asking for a root cell is meaningful.

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import cellrank as cr
import matplotlib.pyplot as plt
from moscot.problems.time import TemporalProblem

print("cellrank", cr.__version__)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, dpi_save=200, frameon=False, figsize=(5, 5))

base = "/project/imoskowitz/yubin/Trajectory_inference/"
fig_cr = base + "output_fig/08_cellrank_joint/"
res_dir = base + "data/cellrank/"
for d in (fig_cr, res_dir):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = fig_cr

BASIS = "X_umap_post"
REP = "X_pca_harmony"
N_STATES = 14

# --- compatibility shim -------------------------------------------------
# scanpy 1.11.5 calls .getnnz() on the kNN distance matrix, but scipy >= 1.16
# hands back a csr_*array* (sparse arrays dropped getnnz), so sc.pp.neighbors
# raises AttributeError. Restore the method rather than pinning scipy, which
# would risk the numpy-2 ABI that petsc4py in this env is built against.
import scipy.sparse as _sp

if not hasattr(_sp.csr_array, "getnnz"):
    def _getnnz(self, axis=None):
        if axis is None:
            return int(self.nnz)
        if axis == 1:
            return np.diff(self.indptr)
        return np.bincount(self.indices, minlength=self.shape[1])
    for _cls in (_sp.csr_array, _sp.csc_array):
        _cls.getnnz = _getnnz
    print("patched sparse-array getnnz for scanpy/scipy compatibility")
# ------------------------------------------------------------------------


def as_sparse_matrices(adata):
    """Force obsp/layers to scipy sparse *matrices*.

    anndata >= 0.12 hands back csr_*arrays*, but scvelo 0.3.4 still uses
    matrix-only APIs (.A1, .getnnz), so it crashes on them. Convert at the
    source instead of patching each call site. Re-run after anything that
    rebuilds these (subsetting, copying).
    """
    from scipy.sparse import csr_matrix, issparse, spmatrix
    for store in (adata.obsp, adata.layers):
        for k in list(store.keys()):
            v = store[k]
            if issparse(v) and not isinstance(v, spmatrix):
                store[k] = csr_matrix(v)
    return adata

In [ ]:
adata = sc.read_h5ad(base + "data/devbrain_brooks_integrated.h5ad")
# the joint graph already exists as the "post" neighbours from the integration run
adata.obsp["connectivities"] = adata.obsp["post_connectivities"]
adata.obsp["distances"] = adata.obsp["post_distances"]
adata.uns["neighbors"] = adata.uns["post"]
as_sparse_matrices(adata)
print(adata)
print(adata.obs.groupby(["dataset", "stage"], observed=True).size().to_string())

## moscot across all nine stages

In [ ]:
tp = TemporalProblem(adata)
tp = tp.prepare(time_key="stage_num", joint_attr={"attr": "obsm", "key": REP})
print(tp)

In [ ]:
tp = tp.solve(epsilon=1e-2, tau_a=0.95, tau_b=0.95, scale_cost="mean")
for k, v in tp.solutions.items():
    print(f"  {k}: converged={v.converged}  cost={float(v.cost):.4g}")

In [ ]:
rtk = cr.kernels.RealTimeKernel.from_moscot(tp)
rtk = rtk.compute_transition_matrix(self_transitions="all", conn_weight=0.2)
ck = cr.kernels.ConnectivityKernel(adata).compute_transition_matrix()
print(rtk, "\n", ck)

KERNELS = {
    "joint_realtime":          rtk,
    "joint_rtk0.8_ck0.2":      0.8 * rtk + 0.2 * ck,
    "joint_rtk0.5_ck0.5":      0.5 * rtk + 0.5 * ck,
}

In [ ]:
def run_estimator(name, kernel, n_states=N_STATES, cluster_key="leiden_joint"):
    print("=" * 70, f"\n{name}\n", "=" * 70, flush=True)
    g = cr.estimators.GPCCA(kernel)
    g.compute_schur(n_components=n_states + 8, method="krylov")
    g.compute_macrostates(n_states=n_states, cluster_key=cluster_key)
    g.predict_terminal_states()
    try:
        g.predict_initial_states(n_states=1)
        init = list(g.initial_states.cat.categories)
    except Exception as e:
        print(f"  initial-state prediction failed: {e}")
        init = []
    print(f"  terminal: {list(g.terminal_states.cat.categories)}")
    print(f"  initial : {init}")

    ms = g.macrostates
    comp = (pd.crosstab(ms[~ms.isna()], adata.obs.loc[~ms.isna(), "stage"],
                        normalize="index") * 100).round(1)
    print("\n  macrostate composition by stage (%):")
    print(comp.to_string())
    ds = (pd.crosstab(ms[~ms.isna()], adata.obs.loc[~ms.isna(), "dataset"],
                      normalize="index") * 100).round(1)
    print("\n  macrostate composition by dataset (%):")
    print(ds.to_string())

    g.compute_fate_probabilities()
    g.plot_macrostates(which="all", basis=BASIS, legend_loc="right",
                       title=f"{name}: macrostates", save=f"{name}_macrostates.png")
    g.plot_macrostates(which="terminal", basis=BASIS, legend_loc="right",
                       title=f"{name}: terminal", save=f"{name}_terminal.png")
    if init:
        g.plot_macrostates(which="initial", basis=BASIS, legend_loc="right",
                           title=f"{name}: initial", save=f"{name}_initial.png")
    return {"name": name, "terminal": list(g.terminal_states.cat.categories),
            "initial": init, "composition": comp, "macrostates": ms,
            "fate": pd.DataFrame(np.asarray(g.fate_probabilities),
                                 columns=list(g.fate_probabilities.names),
                                 index=adata.obs_names)}


results = {}
for name, kern in KERNELS.items():
    try:
        results[name] = run_estimator(name, kern)
    except Exception as e:
        print(f"!! {name} FAILED: {type(e).__name__}: {e}", flush=True)
        results[name] = {"name": name, "error": f"{type(e).__name__}: {e}"}

## Where does the root land?

The point of adding Brooks was to make the earliest stage reachable. If the initial state
sits in E7.0/E7.5 the bridge did its job; if it sits at E8.5+ the OT coupling is not carrying
the early cells, whatever the UMAP looks like.

In [ ]:
rows = []
for name, r in results.items():
    if "error" in r:
        rows.append({"kernel": name, "status": "FAILED", "detail": r["error"][:60]})
        continue
    stage_of = {}
    for which in ("initial", "terminal"):
        for st in r[which]:
            sel = (r["macrostates"] == st).values
            if sel.any():
                stage_of[f"{which}:{st}"] = str(
                    adata.obs.loc[sel, "stage"].value_counts(normalize=True).idxmax())
    rows.append({"kernel": name, "status": "ok",
                 "initial": ", ".join(f"{k.split(':')[1]}@{v}" for k, v in stage_of.items()
                                      if k.startswith("initial")) or "none",
                 "n_terminal": len(r["terminal"])})
print(pd.DataFrame(rows).to_string(index=False))
pd.DataFrame(rows).to_csv(res_dir + "joint_kernel_overview.csv", index=False)

In [ ]:
import resource
print(f"peak RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
for name, r in results.items():
    if "error" not in r:
        r["fate"].to_csv(res_dir + f"fate_{name}.csv")
print("done")